In [2]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from transformers import AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = "../models/Qwen3-0.6B-W4A16"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

print(f"Base model:      {BASE_MODEL}")
print(f"Quantized model: {OUTPUT_DIR}")

Base model:      Qwen/Qwen3-0.6B
Quantized model: ../models/Qwen3-0.6B-W4A16


In [3]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

2026-06-08T01:25:10.1748 | get_main_device | WARNING - No accelerator available! Compressing model on CPU instead
Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False block_size=128 dampening_frac=0.01 actorder=static offload_hessians=False


In [4]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=512,
        num_calibration_samples=32,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

In [20]:
import pathlib
from huggingface_hub import snapshot_download
import os

# Resolve the cached HuggingFace path for the base model automatically
# so this works on any machine without hardcoded Windows paths.
hf_cache = os.path.join(
    os.path.expanduser("~"),
    ".cache", "huggingface", "hub",
    "models--Qwen--Qwen3-0.6B"
)

def folder_size(path):
    p = pathlib.Path(path)
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(nbytes):
    if nbytes < 1024**2:
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3:
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig  = folder_size(hf_cache)
size_quant = folder_size(OUTPUT_DIR)
reduction  = (1 - size_quant / size_orig) * 100

print("Model Size Comparison")
print("=" * 50)
print(f"Original Model : {format_size(size_orig)}")
print(f"Quantized Model: {format_size(size_quant)}")
print(f"Reduction      : {reduction:.2f}%")

Model Size Comparison
Original Model : 2.83 GB
Quantized Model: 528.7 MB
Reduction      : 81.76%


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

BASE_MODEL = "Qwen/Qwen3-0.6B"

prompt = "Machine learning is a branch of"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="cpu",
    dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Base Model ({BASE_MODEL})")
print(tokenizer.decode(generated, skip_special_tokens=True))

Base Model (Qwen/Qwen3-0.6B)
 artificial intelligence that has gained significant attention in recent years, particularly in the context of the rise of big data and the need for efficient, scalable solutions to complex problems. As the field continues to evolve, the integration of machine learning into various industries is becoming increasingly widespread. However, despite its growing popularity,


In [17]:
import logging
from transformers import AutoModelForCausalLM
import torch

logging.getLogger("llmcompressor").setLevel(logging.WARNING)

quant_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    device_map="cpu",
    dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = quant_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Quantized Model ({OUTPUT_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Decompressing model: 100%|███████████████████████████████████████████████████████████| 196/196 [00:04<00:00, 48.04it/s]


Quantized Model (../models/Qwen3-0.6B-W4A16)
Prompt: Machine learning is a branch of
Response:  computer science that uses algorithms to make decisions in a computer. It is used in many applications, such as in the field of computer science, and in the field of the natural sciences. It is also used in the field of the social sciences. In the field of the social sciences, it is used


In [8]:
from datasets import load_dataset

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
print(f"Loaded {len(test_data)} test samples")

Loaded 4358 test samples


In [9]:
quant_ppl = calculate_perplexity(quant_model, tokenizer, test_data)
print(f"Quantized perplexity: {quant_ppl:.2f}")

Quantized perplexity: 36.38


In [23]:
base_model_ppl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)
base_ppl = calculate_perplexity(base_model_ppl, tokenizer, test_data)
print(f"Base (BF16) perplexity: {base_ppl:.2f}")

`torch_dtype` is deprecated! Use `dtype` instead!


Base (BF16) perplexity: 32.76


In [24]:
ppl_diff   = quant_ppl - base_ppl
ppl_change = (ppl_diff / base_ppl) * 100

print("=" * 50)
print("Perplexity Comparison")
print("=" * 50)
print(f"Base (BF16):       {base_ppl:.8f}")
print(f"Quantized (W4A16): {quant_ppl:.8f}")
print(f"Absolute diff:     {ppl_diff:+.8f}")
print(f"Relative change:   {ppl_change:+.4f}%")
print()
print("Size Reduction:    {:.2f}%".format(reduction))

Perplexity Comparison
Base (BF16):       32.75856165
Quantized (W4A16): 36.38363450
Absolute diff:     +3.62507285
Relative change:   +11.0660%

Size Reduction:    81.76%
